# 🦷 Dental AI — Training on Google Colab

**This notebook trains your full dental AI model for free using Colab's GPU.**

Steps:
1. Mount Google Drive (to save models)
2. Install requirements
3. Prepare dataset
4. Train YOLOv8 (detection)
5. Train U-Net (segmentation)
6. Download trained weights

In [ ]:
# ── STEP 1: Check GPU ──────────────────────────────────────────────
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── STEP 2: Mount Google Drive (models saved here) ─────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/DentalAI', exist_ok=True)
print('Drive mounted — models will be saved to: /content/drive/MyDrive/DentalAI/')

In [ ]:
# ── STEP 3: Upload & extract your project ─────────────────────────
# Option A: Upload the zip file
from google.colab import files
uploaded = files.upload()   # upload dental_ai.zip
!unzip -q dental_ai.zip -d /content/
%cd /content/dental_ai

In [ ]:
# ── STEP 4: Install requirements ──────────────────────────────────
!pip install -q ultralytics torch torchvision opencv-python-headless \
               albumentations fastapi uvicorn python-multipart tqdm

In [ ]:
# ── STEP 5: Download real dental dataset from Roboflow ─────────────
# Go to https://universe.roboflow.com
# Search: 'dental xray detection'
# Click Export → YOLOv8 → get download code below:

# Paste your Roboflow download code here:
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY_HERE')
# project = rf.workspace('YOUR_WORKSPACE').project('dental-xray')
# dataset = project.version(1).download('yolov8')

# OR use synthetic data to test the pipeline first:
!python training/dataset_prep.py --synthetic-only
print('Dataset ready!')

In [ ]:
# ── STEP 6: Train YOLOv8 ───────────────────────────────────────────
# This takes ~30 min on Colab T4 GPU
!python training/train_yolo.py --epochs 100 --batch 16 --device cuda

In [ ]:
# ── STEP 7: Train U-Net segmentation ──────────────────────────────
# This takes ~20 min on Colab T4 GPU
!python training/train_unet.py --epochs 50 --batch 8 --device cuda

In [ ]:
# ── STEP 8: Show training results ─────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('models/unet_training_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log['epoch'], log['train_loss'], label='Train loss')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val loss')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(log['epoch'], log['val_iou'], color='green', label='Val IoU')
axes[1].set_title('Validation IoU'); axes[1].legend()
plt.tight_layout(); plt.show()

best_iou = log['val_iou'].max()
print(f'Best val IoU: {best_iou}')

In [ ]:
# ── STEP 9: Save models to Google Drive ───────────────────────────
import shutil
shutil.copy('models/unet_dental_best.pth',
            '/content/drive/MyDrive/DentalAI/unet_dental_best.pth')
shutil.copy('models/yolo_dental/weights/best.pt',
            '/content/drive/MyDrive/DentalAI/yolo_best.pt')
print('Models saved to Google Drive!')

In [ ]:
# ── STEP 10: Download models to your laptop ───────────────────────
from google.colab import files
files.download('models/unet_dental_best.pth')
files.download('models/yolo_dental/weights/best.pt')
print('Download started!')